In [ ]:
# Step 1: Clone the repository (feature/go-phase0-pretraining branch) and install dependencies.
!git clone -b feature/go-phase0-pretraining --single-branch https://github.com/HUBioDataLab/ContVAR.git /content/ContVAR
%cd /content/ContVAR

%pip install -q graphein MDAnalysis torch_geometric torchmetrics wandb biopython h5py
!apt-get -qq install dssp
%pip install -e .

In [ ]:
# Step 2: Mount Google Drive to access data files.
# Drive is only used for two large files (not code):
#   - protein_triplets_data_9march.zip  (CIF structure files)
#   - embeddings_variable.h5            (precomputed ESM2 embeddings)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3: Login to Weights & Biases and set up the environment.
# - wandb.login() will prompt for your API key interactively.
# - setup_environment() auto-detects Colab, locates data/embeddings on Drive
#   for the DMS (protein_triplets_data + embeddings_variable.h5) side.
import wandb
from contvar import setup_environment, train_pipeline, visualize_tsne

wandb.login(key="2becafa4dcb70173759a7b50ee5de92401c637c4")
env = setup_environment()  # env['device'], env['data_root'], env['embeddings_path']

In [ ]:
# Step 4: Run the full training pipeline (Phase 0 + curriculum learning).
# - Phase 0: GO semantic similarity pretraining (MF/BP/CC heads) using semantic_similarity TSVs.
# - Phase 1: Exhaustive triplet training with standard triplet loss.
# - Phase 2: Streaming semi-hard negative mining with online triplet loss.
# - force=True  → generate all graphs from scratch.
# - force=False → reuse previously processed graphs if available.
# - split_path  → path to an existing split.json for reproducibility (None = create new split).

# Colab paths (adjust if you keep data elsewhere):
go_tsv_dir = "/content/drive/MyDrive/ContVAR/semantic_similarity"
# We only provide the ZIP; go_pretraining will extract it under /content/content/alphafold_structures
# and search recursively for CIFs like .../alphafold_structures/cif/<id>.cif

go_structures_zip = "/content/drive/MyDrive/ContVAR/alphafold_structures.zip"

config_overrides = {
    # Enable Phase 0 GO pretraining (set to 0 to disable)
    "go_phase0_epochs": 5,
    "go_tsv_dir": go_tsv_dir,
    "go_structures_zip": go_structures_zip,
    # Let go_structure_root be derived automatically from the zip name
    "go_use_esm_embeddings": False,
}

model = train_pipeline(
    config=config_overrides,
    force=False,
    split_path=None,
    data_root=env['data_root'],
    embeddings_path=env['embeddings_path'],
    device=env['device'],
)

In [ ]:
# Step 5: Visualize learned embeddings using t-SNE.
# Plots interactive scatter plots (Plotly) comparing:
#   - Baseline (raw pooled ESM2 features, no GNN) vs Trained model embeddings
#   - Global (graph-level) vs Local (mutation-position) embeddings
# Each point is colored by protein family, shaped by role (anchor/positive/negative).
visualize_tsne(model=model, splits=['val'], data_root=env['data_root'], device=env['device'],)

In [ ]:
# Step 6: Download trained model checkpoints and data split.
# Note: training.py saves 'model_best_loss.pt' for the best epoch
#       and 'model_last.pt' for the last epoch model.
from google.colab import files

for f in ['model_best_loss.pt', 'model_last.pt', 'split.json']:
    files.download(f)